Project Description

In [ ]:
"""
Financial Time Series Volatility Analysis

This project analyzes volatility and inter-market relationships between:
- Oil Prices (CL=F)
- Exchange Rates (EURUSD=X)
- Dow Jones (^DJI)

Models used:
- ARCH / GARCH (volatility)
- VAR (interdependence)
- VECM (long-run equilibrium)
"""

Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

from arch import arch_model
from statsmodels.tsa.api import VAR
from statsmodels.tsa.vector_ar.vecm import coint_johansen
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox

plt.style.use("seaborn")

Data Collection

In [ ]:
tickers = ["CL=F", "EURUSD=X", "^DJI"]

data = yf.download(tickers, start="2020-01-01", end="2023-12-31")["Adj Close"]
data.dropna(inplace=True)

data.head()

Returns

In [ ]:
returns = data.pct_change().dropna()

returns.plot(figsize=(10,5), title="Returns of Financial Series")
plt.show()

Stationarity Check

In [ ]:
def adf_test(series, name):
    result = adfuller(series)
    print(f"{name} ADF p-value:", result[1])

for col in returns.columns:
    adf_test(returns[col], col)

GARCH Model

In [ ]:
oil_returns = returns["CL=F"] * 100

garch = arch_model(oil_returns, vol="Garch", p=1, q=1)
garch_fit = garch.fit(disp="off")

print(garch_fit.summary())

GARCH Forecast

In [ ]:
garch_forecast = garch_fit.forecast(horizon=90)
volatility = np.sqrt(garch_forecast.variance.values[-1, :])

plt.plot(volatility)
plt.title("GARCH Forecasted Volatility (3 Months)")
plt.show()

ARCH Model (Comparison)

In [ ]:
arch = arch_model(oil_returns, vol="ARCH", p=1)
arch_fit = arch.fit(disp="off")

print(arch_fit.summary())

Ljung-Box Test

In [ ]:
lb_test = acorr_ljungbox(oil_returns, lags=20, return_df=True)
print(lb_test)

VAR Model

In [ ]:
var_model = VAR(returns)

lag_selection = var_model.select_order(maxlags=10)
print(lag_selection.summary())

lag = lag_selection.selected_orders['aic']

var_fit = var_model.fit(lag)
print(var_fit.summary())

VAR Forecast

In [ ]:
forecast = var_fit.forecast(returns.values[-lag:], steps=5)

forecast_df = pd.DataFrame(forecast, columns=returns.columns)
forecast_df

Cointegration (Johansen)

In [ ]:
coint_test = coint_johansen(data, det_order=0, k_ar_diff=1)

print("Trace Statistics:", coint_test.lr1)
print("Critical Values:", coint_test.cvt)

**Interpretation Note**

**Key Observations:**

- GARCH confirms volatility clustering and persistence
- ARCH captures heteroskedasticity but is less effective for forecasting
- Ljung-Box test confirms presence of ARCH/GARCH effects
- VAR shows interdependence between oil, exchange rate, and stock market
- Cointegration suggests long-run equilibrium relationships

This aligns with financial theory where macro variables are interconnected.
